# GPM IMERG monthly precipitation (NASA Earthdata)

Download a monthly GPM IMERG precipitation granule from GES DISC through one Earthdata Login, then map it. This is a **live** query: it needs the `[earthdata]` extra (Python >=3.12) and EDL credentials (`EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` or `~/.netrc`). The query is wrapped so the notebook stays safe under nbval-lax when run offline / without credentials. See [Authentication](../../reference/earthdata/authentication.md) and [Usage](../../reference/earthdata/usage.md).

## Setup

Import the unified `EarthLens` entry point and prepare a local output directory. `download()` writes the fetched granule(s) into `earthdata_output/`.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

## Download the IMERG granule

### Build the request

Configure an `EarthLens` request for the monthly GPM IMERG dataset (`GPM_3IMERGM_07`), one month of June 2023, over a near-global window. Keeping construction on its own line makes the parameters easy to read and tweak.

In [ ]:
imerg = EarthLens(
    data_source='earthdata',
    dataset='GPM_3IMERGM_07',
    variables=['precipitation'],
    start='2023-06-01',
    end='2023-06-30',
    aoi=[-180.0, -60.0, 180.0, 60.0],
    path=str(OUT_DIR),
)

### Fetch it

Run `download()` to fetch the granule(s) to disk. The call is wrapped in a `try`/`except` so the notebook degrades gracefully — printing a skip message instead of erroring — when run offline or without Earthdata credentials.

In [ ]:
paths = None
try:
    paths = imerg.download(progress_bar=False)
    print(len(paths), 'granule(s):', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Map the precipitation field

If the download succeeded, open the granule with `xarray` (the data lives in the `Grid` group) and plot the June 2023 mean precipitation rate. This step is also guarded so a read / plotting surprise does not mask the download above.

In [ ]:
if paths:
    try:
        import matplotlib.pyplot as plt
        import xarray as xr

        ds = xr.open_dataset(paths[0], group='Grid')
        precip = ds['precipitation'].isel(time=0)
        fig, ax = plt.subplots(figsize=(10, 5))
        precip.where(precip >= 0).T.plot(ax=ax, cmap='viridis', robust=True)
        ax.set_title('GPM IMERG monthly precipitation rate, June 2023')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'plot step skipped: {type(exc).__name__}: {exc}')